# Handling Categorical Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('cars.csv')

In [ ]:
df.shape

(8128, 5)

In [ ]:
df.head(5)

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [ ]:
print(df.value_counts('brand'))
print(df.value_counts('fuel'))
print(df.value_counts('owner'))

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Ambassador          4
Kia                 4
Daewoo              3
MG                  3
Ashok               1
Opel                1
Peugeot             1
Name: count, dtype: int64
fuel
Diesel    4402
Petrol    3631
CNG         57
LPG         38
Name: count, dtype: int64
owner
First Owner             5289
Second Owner            2105
Third Owner              555
Fourth & Above Owner     174
Test Drive Car             5
Name: count, dtype: in

# 1. OneHotEncoding using Pandas

In [ ]:
df_dummies = pd.get_dummies(df, columns=['fuel', 'owner'])

# Convert boolean columns to int (0 or 1)
boolean_cols = df_dummies.select_dtypes(include='bool').columns
df_dummies[boolean_cols] = df_dummies[boolean_cols].astype(int)

df_encoded = df_dummies.copy() # Create a copy for further use if needed
display(df_encoded.head())

,brand,km_driven,selling_price,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,145500,450000,0,1,0,0,1,0,0,0,0
1,Skoda,120000,370000,0,1,0,0,0,0,1,0,0
2,Honda,140000,158000,0,0,0,1,0,0,0,0,1
3,Hyundai,127000,225000,0,1,0,0,1,0,0,0,0
4,Maruti,120000,130000,0,0,0,1,1,0,0,0,0


## 2. K-1 OneHotEncoding

In [ ]:
df_dummies=pd.get_dummies(df,columns=['fuel','owner'],drop_first=True)
# Convert boolean columns to int (0 or 1)
boolean_cols = df_dummies.select_dtypes(include='bool').columns
df_dummies[boolean_cols] = df_dummies[boolean_cols].astype(int)

# 3. OneHotEncoding using Sklearn

### 3.1 Train_Test_split

In [ ]:
from sklearn.model_selection import train_test_split
X_train , X_test , Y_train,Y_test = train_test_split(df.drop('selling_price',axis=1),df['selling_price'], test_size=0.2,random_state=42)

In [ ]:
X_train.head()

,brand,km_driven,fuel,owner
6518,Tata,2560,Petrol,First Owner
6144,Honda,80000,Petrol,Second Owner
6381,Hyundai,150000,Diesel,Fourth & Above Owner
438,Maruti,120000,Diesel,Second Owner
5939,Maruti,25000,Petrol,First Owner


In [ ]:
X_train.shape , X_test.shape , Y_test.shape,Y_test.shape

((6502, 4), (1626, 4), (1626,), (1626,))

### 3.2 Importing class of onehot encoding

In [ ]:
from sklearn.preprocessing import OneHotEncoder
# object creation
ohe = OneHotEncoder(drop='first',sparse_output=False,dtype=np.int32)

In [ ]:
X_train_new = ohe.fit_transform(X_train[['fuel','owner']])

In [ ]:
X_test_new = ohe.transform(X_test[['fuel','owner']])

In [ ]:
X_train_new.shape

(6502, 7)

In [ ]:
np.hstack((X_train[['brand','km_driven']].values,X_train_new))

array([['Tata', 2560, 0, ..., 0, 0, 0],
       ['Honda', 80000, 0, ..., 1, 0, 0],
       ['Hyundai', 150000, 1, ..., 0, 0, 0],
       ...,
       ['Hyundai', 35000, 0, ..., 0, 0, 0],
       ['Maruti', 27000, 1, ..., 0, 0, 0],
       ['Maruti', 70000, 0, ..., 1, 0, 0]], dtype=object)

# 4. OneHotEncoding with Top Categories

In [ ]:
counts = df['brand'].value_counts()
counts

,count
brand,
Maruti,2448
Hyundai,1415
Mahindra,772
Tata,734
Toyota,488
Honda,467
Ford,397
Chevrolet,230
Renault,228


In [ ]:
print(df['brand'].nunique())
threshold = 100

32


In [ ]:
repl = counts[counts <= threshold].index

In [ ]:
df_brand_dummies = pd.get_dummies(df['brand'].replace(repl, 'uncommon'))
boolean_cols_brand = df_brand_dummies.select_dtypes(include='bool').columns
df_brand_dummies[boolean_cols_brand] = df_brand_dummies[boolean_cols_brand].astype(int)
display(df_brand_dummies.sample(5))

,BMW,Chevrolet,Ford,Honda,Hyundai,Mahindra,Maruti,Renault,Skoda,Tata,Toyota,Volkswagen,uncommon
5636,0,0,0,0,0,0,1,0,0,0,0,0,0
3277,0,0,0,0,0,0,0,1,0,0,0,0,0
8113,0,0,0,0,0,0,0,0,0,0,1,0,0
5919,0,0,0,1,0,0,0,0,0,0,0,0,0
1420,0,0,1,0,0,0,0,0,0,0,0,0,0
